# 01 - Biclustering

This notebook demonstrates:
1. SpectralBiclustering on ORL training set
2. Block structure visualization (checkerboard heatmap)
3. Cluster size statistics
4. Mean face per cluster
5. assign_row_cluster for test samples

In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from data.load_dataset import load_orl, train_test_split_orl
from src.biclustering import BidirectionalClustering
from src.utils import show_bicluster_heatmap, show_faces

X, y = load_orl(data_dir='../data/ORL')
X_train, X_test, y_train, y_test = train_test_split_orl(X, y, n_train=5)
IMG_SHAPE = (64, 64)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (200, 4096), Test: (200, 4096)


## 1. Fit BidirectionalClustering

In [2]:
bicluster = BidirectionalClustering(n_row_clusters=4, n_col_clusters=4, random_state=42)
bicluster.fit(X_train)
bicluster.summary()
blocks, row_idx_map, col_idx_map = bicluster.get_blocks(X_train)
print(f'Generated {len(blocks)} blocks')

双向聚类结果 (method=spectral)
  行聚类数 k_r = 4
    簇 0: 19 个样本
    簇 1: 51 个样本
    簇 2: 61 个样本
    簇 3: 69 个样本
  列聚类数 k_c = 4
    簇 0: 1069 个特征
    簇 1: 1160 个特征
    簇 2: 246 个特征
    簇 3: 1621 个特征
Generated 16 blocks


## 2. Block Structure Heatmap (Checkerboard)

In [3]:
fig = show_bicluster_heatmap(
    X_train,
    bicluster.row_labels_,
    bicluster.col_labels_,
    title='Bicluster Structure (ORL Training Set)',
    save_path='../results/figures/01_bicluster_heatmap.png'
)
plt.show()

C:\Users\86175\AppData\Local\Temp\ipykernel_42312\1365179323.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Cluster Size Statistics

In [4]:
print("Row cluster sizes:")
for a in range(bicluster.n_row_clusters):
    cnt = (bicluster.row_labels_ == a).sum()
    print(f"  Cluster {a}: {cnt} samples")

print("")
print("Col cluster sizes:")
for b in range(bicluster.n_col_clusters):
    cnt = (bicluster.col_labels_ == b).sum()
    print(f"  Cluster {b}: {cnt} features")

print("")
print("Block sizes:")
for (a, b), blk in sorted(blocks.items()):
    print(f"  Block ({a},{b}): {blk.shape}")

Row cluster sizes:
  Cluster 0: 19 samples
  Cluster 1: 51 samples
  Cluster 2: 61 samples
  Cluster 3: 69 samples

Col cluster sizes:
  Cluster 0: 1069 features
  Cluster 1: 1160 features
  Cluster 2: 246 features
  Cluster 3: 1621 features

Block sizes:
  Block (0,0): (19, 1069)
  Block (0,1): (19, 1160)
  Block (0,2): (19, 246)
  Block (0,3): (19, 1621)
  Block (1,0): (51, 1069)
  Block (1,1): (51, 1160)
  Block (1,2): (51, 246)
  Block (1,3): (51, 1621)
  Block (2,0): (61, 1069)
  Block (2,1): (61, 1160)
  Block (2,2): (61, 246)
  Block (2,3): (61, 1621)
  Block (3,0): (69, 1069)
  Block (3,1): (69, 1160)
  Block (3,2): (69, 246)
  Block (3,3): (69, 1621)


## 4. Mean Face per Row Cluster

In [5]:
# Mean face per row cluster
n_clusters = bicluster.n_row_clusters
fig, axes = plt.subplots(1, n_clusters, figsize=(n_clusters * 2.5, 3))

for a in range(n_clusters):
    mask = bicluster.row_labels_ == a
    n_a = int(mask.sum())
    cluster_mean = X_train[mask].mean(axis=0)
    axes[a].imshow(cluster_mean.reshape(IMG_SHAPE), cmap='gray', vmin=0, vmax=1)
    axes[a].set_title(f'Cluster {a} (n={n_a})', fontsize=9)
    axes[a].axis('off')

fig.suptitle('Mean Face per Row Cluster', fontsize=11)
plt.tight_layout()
plt.savefig('../results/figures/01_cluster_mean_faces.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\86175\AppData\Local\Temp\ipykernel_42312\3242783231.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Test Sample Cluster Assignment

In [6]:
test_labels = bicluster.assign_row_cluster(X_test)
print("Test sample cluster assignments:")
for a in range(bicluster.n_row_clusters):
    cnt_tr = (bicluster.row_labels_ == a).sum()
    cnt_te = (test_labels == a).sum()
    print(f"  Cluster {a}: train={cnt_tr}, test={cnt_te}")

Test sample cluster assignments:
  Cluster 0: train=19, test=19
  Cluster 1: train=51, test=48
  Cluster 2: train=61, test=64
  Cluster 3: train=69, test=69
